# NDJF Objective Subtype Features and Clustering

This notebook builds the first objective JPCZ subtype feature table from ERA5-derived fields and then tests whether the events separate naturally in feature space.

This notebook is for **objective forcing/modifier analysis**, not event detection.

Primary goals:
- compute interpretable event-level features from `del dot u`, `del cross u`, `850 hPa` geopotential-height anomaly, and `850 hPa` temperature-gradient magnitude
- inspect 2D and 3D scatterplots before clustering
- run a first hierarchical clustering experiment on the feature table
- compare the resulting clusters against later subjective interpretation without training on those labels


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = True
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import (
    COASTAL_JAPAN_BOX,
    HOKKAIDO_BOX,
    HOKKAIDO_FRONT_BOX,
    OBJECTIVE_SUBTYPE_DOMAIN,
    PACIFIC_EAST_OF_JAPAN_BOX,
    PACIFIC_FRONT_BOX,
    SEA_OF_JAPAN_BOX,
)
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.subtypes import (
    assign_hierarchical_clusters,
    build_objective_subtype_feature_table,
    compute_mean_silhouette_score,
    compute_monthly_geopotential_height_climatology,
    compute_pca_scores,
    evaluate_hierarchical_cluster_solutions,
    feature_definitions_dataframe,
    standardize_feature_table,
)

REVIEW_CATALOG_PATH = Path(DRIVE_OUTPUT_DIR) / "jpcz_catalog_ndjf_merged_12h_manual_verification.csv"
FEATURE_TABLE_PATH = Path("outputs/verification/jpcz_catalog_ndjf_objective_subtype_features.csv")
CLUSTER_TABLE_PATH = Path("outputs/verification/jpcz_catalog_ndjf_objective_subtype_features_with_clusters.csv")
CLUSTER_QUALITY_PATH = Path("outputs/verification/jpcz_catalog_ndjf_cluster_quality_scan.csv")
FEATURE_DICTIONARY_PATH = Path("outputs/verification/jpcz_catalog_ndjf_objective_feature_dictionary.csv")
CLIMATOLOGY_PATH = Path("outputs/verification/z850_ndjf_monthly_climatology.nc")
SCATTER_DIR = Path("outputs/verification/objective_subtype_scatterplots")
SCATTER_DIR.mkdir(parents=True, exist_ok=True)
RUN_EXPORT_DIR = Path("outputs/verification/objective_subtype_runs")
RUN_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

SUBTYPE_DOMAIN = OBJECTIVE_SUBTYPE_DOMAIN

USE_ONLY_VERIFIED_YES = False
FORCE_REBUILD_Z850_CLIMATOLOGY = False
FORCE_REBUILD_FEATURE_TABLE = False
SAVE_PLOTS = True
ERA5_TIME_CHUNK = 48

CLUSTER_FEATURE_COLUMNS = [
    "coastal_to_jpcz_mean_convergence_ratio",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
    "front_box_max_temp_gradient_850_tminus12_to_tplus12",
    "sea_of_japan_mean_vorticity_peak_925",
]
CLUSTER_COUNT_OPTIONS = [2, 3, 4, 5, 6]
N_CLUSTERS = 4
HIERARCHICAL_METHOD = "ward"

SCATTER_XY_PLOTS = [
    ("coastal_to_jpcz_mean_convergence_ratio", "jpcz_polygon_max_convergence_peak_925"),
    ("coastal_to_jpcz_mean_convergence_ratio", "hokkaido_min_z850_anomaly_tminus12_to_tplus12"),
    ("pacific_to_jpcz_mean_convergence_ratio", "front_box_max_temp_gradient_850_tminus12_to_tplus12"),
    ("sea_of_japan_mean_vorticity_peak_925", "hokkaido_min_z850_anomaly_tminus12_to_tplus12"),
]

FEATURE_DICTIONARY = feature_definitions_dataframe()
FEATURE_UNITS = FEATURE_DICTIONARY.set_index("column_name")["units"].to_dict()


def axis_label(column_name: str) -> str:
    units = FEATURE_UNITS.get(column_name)
    if units is None or units == "unitless":
        return column_name
    return f"{column_name}\n[{units}]"


REGION_TABLE = pd.DataFrame(
    [
        {"name": "Coastal Japan box", **COASTAL_JAPAN_BOX.__dict__},
        {"name": "Pacific east of Japan box", **PACIFIC_EAST_OF_JAPAN_BOX.__dict__},
        {"name": "Hokkaido box", **HOKKAIDO_BOX.__dict__},
        {"name": "Sea of Japan box", **SEA_OF_JAPAN_BOX.__dict__},
        {"name": "Hokkaido front box", **HOKKAIDO_FRONT_BOX.__dict__},
        {"name": "Pacific front box", **PACIFIC_FRONT_BOX.__dict__},
    ]
)


def maybe_copy_to_drive(path: Path):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        print("Copied to Drive:", drive_path)


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True


In [ ]:
# This cell can be rerun independently if earlier config cells were skipped.
from pathlib import Path
import pandas as pd

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import (
    COASTAL_JAPAN_BOX,
    HOKKAIDO_BOX,
    HOKKAIDO_FRONT_BOX,
    PACIFIC_EAST_OF_JAPAN_BOX,
    PACIFIC_FRONT_BOX,
    SEA_OF_JAPAN_BOX,
)
from jpcz_catalog.subtypes import feature_definitions_dataframe

if "DRIVE_OUTPUT_DIR" not in globals():
    DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"
if "REVIEW_CATALOG_PATH" not in globals():
    REVIEW_CATALOG_PATH = Path(DRIVE_OUTPUT_DIR) / "jpcz_catalog_ndjf_merged_12h_manual_verification.csv"
if "USE_ONLY_VERIFIED_YES" not in globals():
    USE_ONLY_VERIFIED_YES = False
if "REGION_TABLE" not in globals():
    REGION_TABLE = pd.DataFrame(
        [
            {"name": "Coastal Japan box", **COASTAL_JAPAN_BOX.__dict__},
            {"name": "Pacific east of Japan box", **PACIFIC_EAST_OF_JAPAN_BOX.__dict__},
            {"name": "Hokkaido box", **HOKKAIDO_BOX.__dict__},
            {"name": "Sea of Japan box", **SEA_OF_JAPAN_BOX.__dict__},
            {"name": "Hokkaido front box", **HOKKAIDO_FRONT_BOX.__dict__},
            {"name": "Pacific front box", **PACIFIC_FRONT_BOX.__dict__},
        ]
    )
if "FEATURE_DICTIONARY" not in globals():
    FEATURE_DICTIONARY = feature_definitions_dataframe()
if "FEATURE_DICTIONARY_PATH" not in globals():
    FEATURE_DICTIONARY_PATH = Path("outputs/verification/jpcz_catalog_ndjf_objective_feature_dictionary.csv")

review_df = pd.read_csv(
    REVIEW_CATALOG_PATH,
    parse_dates=["event_start", "event_end", "event_peak"],
)
review_df = add_japan_local_time_columns(review_df)

if USE_ONLY_VERIFIED_YES:
    working_df = review_df[review_df["verified_event"].fillna("") == "yes"].copy()
else:
    working_df = review_df.copy()

working_df["event_peak"] = pd.to_datetime(working_df["event_peak"])
event_years = sorted(working_df["event_peak"].dt.year.unique())
CLIMATOLOGY_MONTHS = sorted({
    (pd.Timestamp(peak_time) + pd.Timedelta(hours=offset)).month
    for peak_time in working_df["event_peak"]
    for offset in (-12, 0, 12)
})
print("Climatology months needed for t-12/t0/t+12:", CLIMATOLOGY_MONTHS)

FEATURE_DICTIONARY.to_csv(FEATURE_DICTIONARY_PATH, index=False)
if "maybe_copy_to_drive" in globals():
    maybe_copy_to_drive(FEATURE_DICTIONARY_PATH)

print("Total catalog rows:", len(review_df))
print("Rows used for objective subtype features:", len(working_df))
print("Event years represented:", event_years[0], "to", event_years[-1])

print("\nCharacterization regions")
display(REGION_TABLE)

print("\nObjective feature dictionary")
display(FEATURE_DICTIONARY)


In [ ]:
# This cell can recover if the imports/config cell was skipped or deleted.
from pathlib import Path
import shutil

import pandas as pd
import xarray as xr

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import (
    COASTAL_JAPAN_BOX,
    HOKKAIDO_BOX,
    HOKKAIDO_FRONT_BOX,
    OBJECTIVE_SUBTYPE_DOMAIN,
    PACIFIC_EAST_OF_JAPAN_BOX,
    PACIFIC_FRONT_BOX,
    SEA_OF_JAPAN_BOX,
)
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.subtypes import (
    assign_hierarchical_clusters,
    build_objective_subtype_feature_table,
    compute_mean_silhouette_score,
    compute_monthly_geopotential_height_climatology,
    compute_pca_scores,
    evaluate_hierarchical_cluster_solutions,
    feature_definitions_dataframe,
    standardize_feature_table,
)

if "DRIVE_OUTPUT_DIR" not in globals():
    DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"
if "PERSIST_OUTPUTS_TO_DRIVE" not in globals():
    PERSIST_OUTPUTS_TO_DRIVE = True
if "REVIEW_CATALOG_PATH" not in globals():
    REVIEW_CATALOG_PATH = Path(DRIVE_OUTPUT_DIR) / "jpcz_catalog_ndjf_merged_12h_manual_verification.csv"
if "FEATURE_TABLE_PATH" not in globals():
    FEATURE_TABLE_PATH = Path("outputs/verification/jpcz_catalog_ndjf_objective_subtype_features.csv")
if "CLUSTER_TABLE_PATH" not in globals():
    CLUSTER_TABLE_PATH = Path("outputs/verification/jpcz_catalog_ndjf_objective_subtype_features_with_clusters.csv")
if "CLUSTER_QUALITY_PATH" not in globals():
    CLUSTER_QUALITY_PATH = Path("outputs/verification/jpcz_catalog_ndjf_cluster_quality_scan.csv")
if "FEATURE_DICTIONARY_PATH" not in globals():
    FEATURE_DICTIONARY_PATH = Path("outputs/verification/jpcz_catalog_ndjf_objective_feature_dictionary.csv")
if "CLIMATOLOGY_PATH" not in globals():
    CLIMATOLOGY_PATH = Path("outputs/verification/z850_ndjf_monthly_climatology.nc")
if "SCATTER_DIR" not in globals():
    SCATTER_DIR = Path("outputs/verification/objective_subtype_scatterplots")
    SCATTER_DIR.mkdir(parents=True, exist_ok=True)
if "USE_ONLY_VERIFIED_YES" not in globals():
    USE_ONLY_VERIFIED_YES = False
if "FORCE_REBUILD_Z850_CLIMATOLOGY" not in globals():
    FORCE_REBUILD_Z850_CLIMATOLOGY = False
if "FORCE_REBUILD_FEATURE_TABLE" not in globals():
    FORCE_REBUILD_FEATURE_TABLE = False
if "SAVE_PLOTS" not in globals():
    SAVE_PLOTS = True
if "ERA5_TIME_CHUNK" not in globals():
    ERA5_TIME_CHUNK = 48
if "CLUSTER_FEATURE_COLUMNS" not in globals():
    CLUSTER_FEATURE_COLUMNS = [
        "coastal_to_jpcz_mean_convergence_ratio",
        "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
        "front_box_max_temp_gradient_850_tminus12_to_tplus12",
        "sea_of_japan_mean_vorticity_peak_925",
    ]
if "CLUSTER_COUNT_OPTIONS" not in globals():
    CLUSTER_COUNT_OPTIONS = [2, 3, 4, 5, 6]
if "N_CLUSTERS" not in globals():
    N_CLUSTERS = 4
if "HIERARCHICAL_METHOD" not in globals():
    HIERARCHICAL_METHOD = "ward"
if "SCATTER_XY_PLOTS" not in globals():
    SCATTER_XY_PLOTS = [
        ("coastal_to_jpcz_mean_convergence_ratio", "jpcz_polygon_max_convergence_peak_925"),
        ("coastal_to_jpcz_mean_convergence_ratio", "hokkaido_min_z850_anomaly_tminus12_to_tplus12"),
        ("pacific_to_jpcz_mean_convergence_ratio", "front_box_max_temp_gradient_850_tminus12_to_tplus12"),
        ("sea_of_japan_mean_vorticity_peak_925", "hokkaido_min_z850_anomaly_tminus12_to_tplus12"),
    ]
if "maybe_copy_to_drive" not in globals():
    def maybe_copy_to_drive(path: Path):
        if not PERSIST_OUTPUTS_TO_DRIVE:
            return
        drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
        if path.is_file():
            shutil.copy2(path, drive_path)
            print("Copied to Drive:", drive_path)
if "FEATURE_DICTIONARY" not in globals():
    FEATURE_DICTIONARY = feature_definitions_dataframe()
if "FEATURE_UNITS" not in globals():
    FEATURE_UNITS = FEATURE_DICTIONARY.set_index("column_name")["units"].to_dict()
if "axis_label" not in globals():
    def axis_label(column_name: str) -> str:
        units = FEATURE_UNITS.get(column_name)
        if units is None or units == "unitless":
            return column_name
        return f"{column_name}\n[{units}]"
if "REGION_TABLE" not in globals():
    REGION_TABLE = pd.DataFrame(
        [
            {"name": "Coastal Japan box", **COASTAL_JAPAN_BOX.__dict__},
            {"name": "Pacific east of Japan box", **PACIFIC_EAST_OF_JAPAN_BOX.__dict__},
            {"name": "Hokkaido box", **HOKKAIDO_BOX.__dict__},
            {"name": "Sea of Japan box", **SEA_OF_JAPAN_BOX.__dict__},
            {"name": "Hokkaido front box", **HOKKAIDO_FRONT_BOX.__dict__},
            {"name": "Pacific front box", **PACIFIC_FRONT_BOX.__dict__},
        ]
    )

if "working_df" not in globals() or "event_years" not in globals():
    review_df = pd.read_csv(
        REVIEW_CATALOG_PATH,
        parse_dates=["event_start", "event_end", "event_peak"],
    )
    review_df = add_japan_local_time_columns(review_df)
    if USE_ONLY_VERIFIED_YES:
        working_df = review_df[review_df["verified_event"].fillna("") == "yes"].copy()
    else:
        working_df = review_df.copy()
    working_df["event_peak"] = pd.to_datetime(working_df["event_peak"])
    event_years = sorted(working_df["event_peak"].dt.year.unique())
    CLIMATOLOGY_MONTHS = sorted({
        (pd.Timestamp(peak_time) + pd.Timedelta(hours=offset)).month
        for peak_time in working_df["event_peak"]
        for offset in (-12, 0, 12)
    })

ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})

if not CLIMATOLOGY_PATH.exists() and "restore_from_drive_cache" in globals():
    restore_from_drive_cache(CLIMATOLOGY_PATH)

if CLIMATOLOGY_PATH.exists() and not FORCE_REBUILD_Z850_CLIMATOLOGY:
    z850_climatology = xr.open_dataarray(CLIMATOLOGY_PATH).load()
    cached_months = {int(month_value) for month_value in z850_climatology["month"].values.tolist()}
else:
    z850_climatology = None
    cached_months = set()

missing_months = sorted(set(CLIMATOLOGY_MONTHS) - cached_months)
if missing_months:
    print("Cached climatology missing months:", missing_months)
    print("Computing missing climatology months one at a time and checkpointing each month to Drive.")
    for month in missing_months:
        month_climatology = compute_monthly_geopotential_height_climatology(
            ds,
            years=event_years,
            months=[month],
            level=850,
        )
        if z850_climatology is None:
            z850_climatology = month_climatology
        else:
            z850_climatology = xr.concat([z850_climatology, month_climatology], dim="month").sortby("month")
        z850_climatology.to_netcdf(CLIMATOLOGY_PATH)
        maybe_copy_to_drive(CLIMATOLOGY_PATH)
        print(f"Checkpointed climatology after month {month:02d}")
    climatology_source = Path("month-by-month climatology checkpoints")
else:
    climatology_source = Path("restored cached climatology")

print("Z850 climatology source:", climatology_source)
print("Required climatology months:", CLIMATOLOGY_MONTHS)
print("Available climatology months:", z850_climatology["month"].values.tolist())
display((z850_climatology.mean(dim=("latitude", "longitude")).to_series()).rename("domain_mean_gpm"))


In [ ]:
if not FEATURE_TABLE_PATH.exists() and "restore_from_drive_cache" in globals():
    restore_from_drive_cache(FEATURE_TABLE_PATH)

if FEATURE_TABLE_PATH.exists() and not FORCE_REBUILD_FEATURE_TABLE:
    feature_df = pd.read_csv(
        FEATURE_TABLE_PATH,
        parse_dates=["event_start", "event_end", "event_peak"],
    )
    feature_source = Path("restored cached feature table")
else:
    feature_df = build_objective_subtype_feature_table(
        ds,
        working_df,
        z850_climatology=z850_climatology,
        progress_every=10,
    )
    feature_df.to_csv(FEATURE_TABLE_PATH, index=False)
    maybe_copy_to_drive(FEATURE_TABLE_PATH)
    feature_source = Path("recomputed from ERA5 feature helpers")

print("Feature table source:", feature_source)
print("Feature rows:", len(feature_df))
preview_columns = [
    "event_peak",
    "event_peak_jst",
    "event_peak_D_1e5_s-1",
    "duration_hours",
    "verified_event",
    "pattern_type_manual",
] + CLUSTER_FEATURE_COLUMNS

display(feature_df.reindex(columns=preview_columns).head(20))


In [ ]:
summary_columns = [
    "jpcz_polygon_mean_convergence_peak_925",
    "jpcz_polygon_max_convergence_peak_925",
    "coastal_japan_mean_convergence_peak_925",
    "coastal_to_jpcz_mean_convergence_ratio",
    "pacific_to_jpcz_mean_convergence_ratio",
    "sea_of_japan_mean_vorticity_peak_925",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
    "sea_of_japan_min_z850_anomaly_tminus12_to_tplus12",
    "front_box_max_temp_gradient_850_tminus12_to_tplus12",
    "pacific_box_max_temp_gradient_850_tminus12_to_tplus12",
]

display(feature_df.loc[:, summary_columns].describe().T)


In [ ]:
color_column = "pattern_type_manual" if "pattern_type_manual" in feature_df.columns else None
plot_df = feature_df.copy()

if color_column is not None:
    plot_df[color_column] = plot_df[color_column].fillna("").replace("", "unlabeled")
    categories = sorted(plot_df[color_column].unique())
    cmap = plt.get_cmap("tab10")
    color_map = {label: cmap(i % 10) for i, label in enumerate(categories)}
else:
    categories = ["all events"]
    color_map = {"all events": "tab:blue"}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.ravel()

for ax, (x_name, y_name) in zip(axes, SCATTER_XY_PLOTS):
    if color_column is None:
        ax.scatter(plot_df[x_name], plot_df[y_name], s=35, alpha=0.75, color="tab:blue")
    else:
        for label in categories:
            subset = plot_df[plot_df[color_column] == label]
            ax.scatter(
                subset[x_name],
                subset[y_name],
                s=35,
                alpha=0.75,
                color=color_map[label],
                label=label,
            )
    ax.set_xlabel(axis_label(x_name))
    ax.set_ylabel(axis_label(y_name))
    ax.grid(alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="lower center", ncol=min(4, len(labels)), bbox_to_anchor=(0.5, -0.02))
fig.suptitle("First-pass objective subtype scatterplots", y=1.02)
fig.tight_layout()

if SAVE_PLOTS:
    scatter_path = SCATTER_DIR / "objective_subtype_scatterplots.png"
    fig.savefig(scatter_path, dpi=170, bbox_inches="tight")
    maybe_copy_to_drive(scatter_path)

plt.show()


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

cluster_input_df = feature_df.copy()
standardized_df, feature_means, feature_stds = standardize_feature_table(
    cluster_input_df,
    columns=CLUSTER_FEATURE_COLUMNS,
)

cluster_quality_df = evaluate_hierarchical_cluster_solutions(
    standardized_df,
    cluster_counts=CLUSTER_COUNT_OPTIONS,
    method=HIERARCHICAL_METHOD,
)
cluster_quality_df.to_csv(CLUSTER_QUALITY_PATH, index=False)
maybe_copy_to_drive(CLUSTER_QUALITY_PATH)

pca_scores, explained_variance_ratio = compute_pca_scores(standardized_df, n_components=3)
cluster_labels = assign_hierarchical_clusters(
    standardized_df,
    n_clusters=N_CLUSTERS,
    method=HIERARCHICAL_METHOD,
)
selected_silhouette = compute_mean_silhouette_score(standardized_df, cluster_labels)

clustered_df = feature_df.copy()
for column_name in pca_scores.columns:
    clustered_df[column_name] = pd.NA
clustered_df.loc[pca_scores.index, pca_scores.columns] = pca_scores
clustered_df[cluster_labels.name] = pd.NA
clustered_df.loc[cluster_labels.index, cluster_labels.name] = cluster_labels.astype(int)

clustered_df.to_csv(CLUSTER_TABLE_PATH, index=False)
maybe_copy_to_drive(CLUSTER_TABLE_PATH)

cluster_col = cluster_labels.name
k_now = int(cluster_col.split("_")[-1])
interpret_cols = [
    "jpcz_polygon_mean_convergence_peak_925",
    "jpcz_polygon_max_convergence_peak_925",
    "coastal_japan_mean_convergence_peak_925",
    "coastal_to_jpcz_mean_convergence_ratio",
    "pacific_to_jpcz_mean_convergence_ratio",
    "sea_of_japan_mean_vorticity_peak_925",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
    "sea_of_japan_min_z850_anomaly_tminus12_to_tplus12",
    "front_box_max_temp_gradient_850_tminus12_to_tplus12",
    "pacific_box_max_temp_gradient_850_tminus12_to_tplus12",
    "duration_hours",
    "event_peak_D_1e5_s-1",
]
cluster_medians = clustered_df.groupby(cluster_col)[interpret_cols].median().round(2)

cluster_medians_path = RUN_EXPORT_DIR / f"cluster_medians_k{k_now}.csv"
clustered_run_path = RUN_EXPORT_DIR / f"clustered_events_k{k_now}.csv"
cluster_quality_run_path = RUN_EXPORT_DIR / f"cluster_quality_scan_k{k_now}.csv"

cluster_medians.to_csv(cluster_medians_path)
clustered_df.to_csv(clustered_run_path, index=False)
cluster_quality_df.to_csv(cluster_quality_run_path, index=False)
maybe_copy_to_drive(cluster_medians_path)
maybe_copy_to_drive(clustered_run_path)
maybe_copy_to_drive(cluster_quality_run_path)

print("Cluster feature columns:", CLUSTER_FEATURE_COLUMNS)
print("Feature means used for standardization:")
display(feature_means.rename("mean"))
print("\nFeature standard deviations used for standardization:")
display(feature_stds.rename("std"))
print("\nCluster-count scan")
display(cluster_quality_df)
print("\nExplained variance ratio:")
for idx, value in enumerate(explained_variance_ratio, start=1):
    print(f"PC{idx}: {value:.3f}")

print(f"\nSelected solution silhouette score: {selected_silhouette:.3f}")
print("\nCluster counts")
display(clustered_df[cluster_col].value_counts(dropna=False).sort_index())
print(f"\nCluster medians (k={k_now})")
display(cluster_medians)

if "pattern_type_manual" in clustered_df.columns:
    print("\nCluster vs subjective pattern cross-tab")
    display(
        pd.crosstab(
            clustered_df[cluster_col],
            clustered_df["pattern_type_manual"].fillna("").replace("", "unlabeled"),
        )
    )

print(f"\nSaved run-specific tables for k={k_now} to: {RUN_EXPORT_DIR}")

fig, ax = plt.subplots(figsize=(9, 7))
for cluster_id in sorted(cluster_labels.dropna().unique()):
    subset = clustered_df[clustered_df[cluster_col] == cluster_id]
    ax.scatter(
        subset["PC1"],
        subset["PC2"],
        s=50,
        alpha=0.8,
        label=f"Cluster {int(cluster_id)}",
    )
ax.set_xlabel(f"PC1 ({explained_variance_ratio[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({explained_variance_ratio[1]:.1%} variance)")
ax.set_title(
    f"Hierarchical clustering in PCA space ({HIERARCHICAL_METHOD}, k={k_now}, silhouette={selected_silhouette:.2f})"
)
ax.grid(alpha=0.25)
ax.legend(loc="best")
fig.tight_layout()

if SAVE_PLOTS:
    pca_path = SCATTER_DIR / f"objective_subtype_pca_clusters_{HIERARCHICAL_METHOD}_k{k_now}.png"
    fig.savefig(pca_path, dpi=170, bbox_inches="tight")
    maybe_copy_to_drive(pca_path)

plt.show()

fig = plt.figure(figsize=(10, 8))
ax3d = fig.add_subplot(111, projection="3d")
for cluster_id in sorted(cluster_labels.dropna().unique()):
    subset = clustered_df[clustered_df[cluster_col] == cluster_id]
    ax3d.scatter(
        subset[CLUSTER_FEATURE_COLUMNS[0]],
        subset[CLUSTER_FEATURE_COLUMNS[1]],
        subset[CLUSTER_FEATURE_COLUMNS[2]],
        s=40,
        alpha=0.8,
        label=f"Cluster {int(cluster_id)}",
    )
ax3d.set_xlabel(axis_label(CLUSTER_FEATURE_COLUMNS[0]))
ax3d.set_ylabel(axis_label(CLUSTER_FEATURE_COLUMNS[1]))
ax3d.set_zlabel(axis_label(CLUSTER_FEATURE_COLUMNS[2]))
ax3d.set_title(f"First-pass 3D objective subtype feature space (k={k_now})")
ax3d.legend(loc="best")
fig.tight_layout()

if SAVE_PLOTS:
    scatter3d_path = SCATTER_DIR / f"objective_subtype_3d_scatter_k{k_now}.png"
    fig.savefig(scatter3d_path, dpi=170, bbox_inches="tight")
    maybe_copy_to_drive(scatter3d_path)

plt.show()
